# 01 · Pobreza infantil

**Fuente:** INE, Encuesta Continua de Hogares (ECH) 2024 y 2025 (implantación), microdatos. Unas 55.000 personas por año.

**Antes de correr:** descargar los microdatos del INE y subir los dos CSV al volume `/Volumes/workspace/pobreza/raw/` (ver `00_setup`).

| Variable | Qué es |
|---|---|
| `e27` | Edad |
| `dpto` | Departamento (código 1 a 19) |
| `pobre17` | Hogar bajo la línea de pobreza, metodología INE 2017 (1 = pobre) |
| `W_ANO` | Ponderador anual |

Pipeline: **bronze** (CSV tal cual) → **silver** (variables clave, dos años unidos) → **gold** (tasas ponderadas) → `data/*.json`

## 1. Bronze

In [0]:
RAW = "/Volumes/workspace/pobreza/raw"

def leer(ruta):
    return (spark.read.option("header", True).option("inferSchema", True)
            .option("sep", ",").option("encoding", "UTF-8").csv(ruta))

ech = {2024: leer(f"{RAW}/ECH_2024_final.csv"), 2025: leer(f"{RAW}/ECH_2025_implantacion.csv")}

for anio, df in ech.items():
    (df.drop("_c0").write.mode("overwrite").option("overwriteSchema", True)
       .saveAsTable(f"workspace.pobreza.ech_{anio}_bronze"))
    print(f"ech_{anio}_bronze:", df.count(), "filas,", len(df.columns), "columnas")

## 2. Silver

- Solo las variables que se usan.
- Se unen 2024 y 2025.
- Los nombres de departamento se ponen a mano desde el código, porque el CSV del INE trae los acentos rotos ("Paysand�").

In [0]:
from pyspark.sql import functions as F

NOMBRES = {1: "Montevideo", 2: "Artigas", 3: "Canelones", 4: "Cerro Largo", 5: "Colonia",
           6: "Durazno", 7: "Flores", 8: "Florida", 9: "Lavalleja", 10: "Maldonado",
           11: "Paysandú", 12: "Río Negro", 13: "Rivera", 14: "Rocha", 15: "Salto",
           16: "San José", 17: "Soriano", 18: "Tacuarembó", 19: "Treinta y Tres"}
mapa = F.create_map([F.lit(x) for kv in NOMBRES.items() for x in kv])

cols = ["ID", "nper", "anio", "mes", "dpto", "e26", "e27", "pobre17", "indig17", "W_ANO"]

silver = (spark.table("workspace.pobreza.ech_2024_bronze").select(cols)
          .unionByName(spark.table("workspace.pobreza.ech_2025_bronze").select(cols))
          .withColumn("nom_dpto", mapa[F.col("dpto")]))

silver.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.pobreza.personas_silver")
display(silver.groupBy("anio", "pobre17").count().orderBy("anio", "pobre17"))

## 3. Gold + export

In [0]:
import os

def carpeta_data():
    """Carpeta data/ del repo, al lado de este notebook."""
    ruta_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    ruta = os.path.dirname(ruta_nb)
    if not ruta.startswith("/Workspace"):
        ruta = "/Workspace" + ruta
    if os.path.basename(ruta) == "notebooks":   # los notebooks viven en notebooks/, los JSON en data/ (raíz)
        ruta = os.path.dirname(ruta)
    os.makedirs(f"{ruta}/data", exist_ok=True)
    return f"{ruta}/data"

def guardar_gold(df, nombre):
    """Guarda la tabla gold en Delta y la exporta a data/<nombre>.json para la web."""
    (df.write.mode("overwrite").option("overwriteSchema", True)
       .saveAsTable(f"workspace.pobreza.{nombre}_gold"))
    df.toPandas().to_json(f"{carpeta_data()}/{nombre}.json", orient="records", force_ascii=False, indent=1)
    print("OK:", nombre)

In [0]:
p = spark.table("workspace.pobreza.personas_silver")

def tasa(df, *by):
    """% de personas en hogares pobres, ponderado, y cantidad de casos en la muestra."""
    return (df.groupBy(*by).agg(
        F.round(F.sum(F.col("pobre17") * F.col("W_ANO")) / F.sum("W_ANO") * 100, 1).alias("pct_pobreza"),
        F.count("*").alias("n_muestra")))

tramo = (F.when(F.col("e27") < 6, "0-5").when(F.col("e27") < 13, "6-12")
          .when(F.col("e27") < 18, "13-17").when(F.col("e27") < 65, "18-64").otherwise("65+"))
orden = (F.when(F.col("e27") < 6, 1).when(F.col("e27") < 13, 2)
          .when(F.col("e27") < 18, 3).when(F.col("e27") < 65, 4).otherwise(5))

# % de pobreza por tramo de edad
guardar_gold(tasa(p.withColumn("tramo", tramo).withColumn("orden", orden), "anio", "orden", "tramo")
             .orderBy("anio", "orden"), "pobreza_por_tramo")
# % de pobreza edad por edad (curva)
guardar_gold(tasa(p.filter(F.col("e27") <= 90), "anio", "e27").withColumnRenamed("e27", "edad")
             .orderBy("anio", "edad"), "pobreza_por_edad")
# % de pobreza infantil (0-17) por departamento
guardar_gold(tasa(p.filter(F.col("e27") < 18), "anio", "dpto", "nom_dpto")
             .orderBy("anio", "dpto"), "pobreza_infantil_dpto")

display(spark.table("workspace.pobreza.pobreza_por_tramo_gold"))

## Hallazgos

- En 2024, el **32,2%** de los niños de 0 a 5 años vivía en hogares pobres, contra el **6,3%** de los mayores de 65: unas 5 veces más.
- La pobreza baja casi sin interrupciones con la edad.
- Por departamento, Artigas, Cerro Largo y Rivera tienen la pobreza infantil más alta (2024). Ojo: las muestras por departamento son chicas.